In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
INDICATORS = ["euribor_3m", "euribor_6m"]

w_seq = Window.partitionBy("indicator").orderBy("date")

# Add monthly change per indicator in basis points (rates in %, *100 = bps)
df_long = (
    spark.table("stocks.bronze.macro_rates")
    .withColumn("prev_value",       F.lag("value", 1).over(w_seq))
    .withColumn("monthly_change_bps", (F.col("value") - F.col("prev_value")) * 100)
    .drop("prev_value", "indicator_label", "ingested_at")
)

# Pivot to wide: one row per date, one column per indicator + change
pivot_value  = df_long.groupBy("date").pivot("indicator", INDICATORS).agg(F.first("value"))
pivot_change = df_long.groupBy("date").pivot("indicator", INDICATORS).agg(F.first("monthly_change_bps"))

for ind in INDICATORS:
    pivot_change = pivot_change.withColumnRenamed(ind, f"{ind}_bps_change")

silver_df = pivot_value.join(pivot_change, on="date", how="left").orderBy("date")

In [0]:
(
    silver_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("stocks.silver.macro_rates")
)

print(f"stocks.silver.macro_rates: {spark.table('stocks.silver.macro_rates').count()} rows")